# 07 — Integração das Bases Curated

## Projeto AgroESG — Soja | Centro-Oeste e Sul | 2019–2024

### Objetivo

Integrar as bases agroambientais previamente tratadas e validadas utilizando
como chave principal:

- `codigo_ibge`
- `ano`

### Bases utilizadas

1. IBGE/PAM — produção municipal de soja;
2. MapBiomas Solo — indicadores de carbono do solo;
3. INMET — indicadores climáticos municipalizados;
4. MapBiomas Cobertura — cobertura e uso da terra.

### Estratégia

A base PAM + MapBiomas Solo previamente integrada será utilizada como núcleo
da análise, pois representa o universo de interesse do projeto: municípios
com produção de soja.

As demais bases serão incorporadas por `codigo_ibge + ano`, preservando
indicadores de qualidade e rastreabilidade.

A integração não representa, por si só, geração ou certificação de créditos
de carbono. O objetivo é construir uma base agroambiental consolidada para
análises posteriores de risco, desempenho e priorização.

In [1]:
# ============================================================
# IMPORTS E CONFIGURAÇÃO
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

pd.set_option(
    "display.width",
    None
)


BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


CURATED_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
)


PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
)


print(
    "BASE_DIR:"
)

print(
    BASE_DIR
)


print(
    "\nCURATED existe?"
)

print(
    CURATED_DIR.exists()
)

BASE_DIR:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao

CURATED existe?
True


In [2]:
# ============================================================
# LOCALIZAÇÃO DAS BASES CURATED
# ============================================================

PADROES_ARQUIVOS = {
    
    "pam_solo": (
        "pam_mapbiomas_solo_integrado_soja_centro_oeste_sul_2019_2024.csv"
    ),

    "inmet": (
        "inmet_municipio_ano_2019_2024.csv"
    ),

    "mapbiomas_cobertura": (
        "mapbiomas_cobertura_municipio_ano_2019_2024.csv"
    )
}


arquivos_encontrados = {}


for nome_base, nome_arquivo in PADROES_ARQUIVOS.items():

    encontrados = list(
        BASE_DIR.rglob(
            nome_arquivo
        )
    )

    arquivos_encontrados[
        nome_base
    ] = encontrados

    print(
        f"\n{nome_base}:"
    )

    print(
        f"Quantidade encontrada: {len(encontrados)}"
    )

    for arquivo in encontrados:

        print(
            " -",
            arquivo
        )


pam_solo:
Quantidade encontrada: 1
 - C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\integracao_pam_mapbiomas_solo\pam_mapbiomas_solo_integrado_soja_centro_oeste_sul_2019_2024.csv

inmet:
Quantidade encontrada: 1
 - C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_2019_2024.csv

mapbiomas_cobertura:
Quantidade encontrada: 1
 - C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\mapbiomas_cobertura\mapbiomas_cobertura_municipio_ano_2019_2024.csv


In [3]:
# ============================================================
# CAMINHOS OFICIAIS DAS BASES
# ============================================================

ARQUIVO_PAM_SOLO = (
    arquivos_encontrados[
        "pam_solo"
    ][0]
)


ARQUIVO_INMET = (
    arquivos_encontrados[
        "inmet"
    ][0]
)


ARQUIVO_MAPBIOMAS_COBERTURA = (
    arquivos_encontrados[
        "mapbiomas_cobertura"
    ][0]
)


print(
    "PAM + SOLO:"
)

print(
    ARQUIVO_PAM_SOLO
)


print(
    "\nINMET:"
)

print(
    ARQUIVO_INMET
)


print(
    "\nMAPBIOMAS COBERTURA:"
)

print(
    ARQUIVO_MAPBIOMAS_COBERTURA
)

PAM + SOLO:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\integracao_pam_mapbiomas_solo\pam_mapbiomas_solo_integrado_soja_centro_oeste_sul_2019_2024.csv

INMET:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_2019_2024.csv

MAPBIOMAS COBERTURA:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\mapbiomas_cobertura\mapbiomas_cobertura_municipio_ano_2019_2024.csv


In [4]:
# ============================================================
# CARREGAMENTO DAS BASES CURATED
# ============================================================

pam_solo = pd.read_csv(
    ARQUIVO_PAM_SOLO,
    dtype={
        "codigo_ibge": "string"
    }
)


inmet = pd.read_csv(
    ARQUIVO_INMET,
    dtype={
        "codigo_ibge": "string"
    }
)


mapbiomas_cobertura = pd.read_csv(
    ARQUIVO_MAPBIOMAS_COBERTURA,
    dtype={
        "codigo_ibge": "string"
    }
)


print(
    "Bases carregadas com sucesso."
)

Bases carregadas com sucesso.


In [5]:
# ============================================================
# DIMENSÕES DAS BASES
# ============================================================

print(
    "PAM + SOLO:"
)

print(
    pam_solo.shape
)


print(
    "\nINMET:"
)

print(
    inmet.shape
)


print(
    "\nMAPBIOMAS COBERTURA:"
)

print(
    mapbiomas_cobertura.shape
)

PAM + SOLO:
(8674, 16)

INMET:
(9966, 30)

MAPBIOMAS COBERTURA:
(9960, 41)


In [6]:
# ============================================================
# COLUNAS DAS BASES
# ============================================================

print(
    "COLUNAS — PAM + SOLO"
)

print(
    pam_solo.columns.tolist()
)


print(
    "\n" + "=" * 80 + "\n")


print(
    "COLUNAS — INMET"
)

print(
    inmet.columns.tolist()
)


print(
    "\n" + "=" * 80 + "\n")


print(
    "COLUNAS — MAPBIOMAS COBERTURA"
)

print(
    mapbiomas_cobertura.columns.tolist()
)

COLUNAS — PAM + SOLO
['codigo_ibge', 'municipio', 'uf', 'regiao', 'ano', 'cultura', 'area_plantada_ha', 'area_colhida_ha', 'area_nao_colhida_ha', 'aproveitamento_area_pct', 'quantidade_produzida_t', 'rendimento_medio_kg_ha', 'valor_producao_mil_reais', 'area_km2', 'carbono_solo_t_ha', 'status_dado']


COLUNAS — INMET
['codigo_ibge', 'municipio', 'uf', 'regiao', 'ano', 'precipitacao_anual_mm', 'origem_precipitacao', 'qualidade_espacial_precipitacao', 'numero_estacoes_observadas_precipitacao', 'cobertura_observada_precipitacao_pct', 'numero_estacoes_idw_precipitacao', 'distancia_max_idw_precipitacao_km', 'temperatura_media_anual_c', 'origem_temperatura', 'qualidade_espacial_temperatura', 'numero_estacoes_observadas_temperatura', 'cobertura_observada_temperatura_pct', 'numero_estacoes_idw_temperatura', 'distancia_max_idw_temperatura_km', 'umidade_media_anual_pct', 'origem_umidade', 'qualidade_espacial_umidade', 'numero_estacoes_observadas_umidade', 'cobertura_observada_umidade_pct', 'nume

In [7]:
# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(
    "PAM + SOLO"
)

display(
    pam_solo.head()
)


print(
    "\nINMET"
)

display(
    inmet.head()
)


print(
    "\nMAPBIOMAS COBERTURA"
)

display(
    mapbiomas_cobertura.head()
)

PAM + SOLO


,codigo_ibge,municipio,uf,regiao,ano,cultura,area_plantada_ha,area_colhida_ha,area_nao_colhida_ha,aproveitamento_area_pct,quantidade_produzida_t,rendimento_medio_kg_ha,valor_producao_mil_reais,area_km2,carbono_solo_t_ha,status_dado
0,4100103,Abatiá,PR,Sul,2019,soja,10570.0,10570.0,0.0,100.0,36784.0,3480.0,43387.0,228.717,55.173093,disponivel
1,4100103,Abatiá,PR,Sul,2020,soja,10470.0,10470.0,0.0,100.0,36435.0,3480.0,50268.0,228.717,55.169301,disponivel
2,4100103,Abatiá,PR,Sul,2021,soja,10220.0,10220.0,0.0,100.0,27727.0,2713.0,69682.0,228.717,55.039162,disponivel
3,4100103,Abatiá,PR,Sul,2022,soja,10470.0,10470.0,0.0,100.0,37064.0,3540.0,114457.0,228.717,55.024834,disponivel
4,4100103,Abatiá,PR,Sul,2023,soja,9960.0,9960.0,0.0,100.0,35856.0,3600.0,94498.0,228.717,55.025820,disponivel



INMET


,codigo_ibge,municipio,uf,regiao,ano,precipitacao_anual_mm,origem_precipitacao,qualidade_espacial_precipitacao,numero_estacoes_observadas_precipitacao,cobertura_observada_precipitacao_pct,numero_estacoes_idw_precipitacao,distancia_max_idw_precipitacao_km,temperatura_media_anual_c,origem_temperatura,qualidade_espacial_temperatura,numero_estacoes_observadas_temperatura,cobertura_observada_temperatura_pct,numero_estacoes_idw_temperatura,distancia_max_idw_temperatura_km,umidade_media_anual_pct,origem_umidade,qualidade_espacial_umidade,numero_estacoes_observadas_umidade,cobertura_observada_umidade_pct,numero_estacoes_idw_umidade,distancia_max_idw_umidade_km,quantidade_variaveis_observadas,tipo_representacao_climatica,score_qualidade_climatica,qualidade_climatica_geral
0,5219902,São Francisco de Goiás,GO,centro_oeste,2019,1073.612878,estimado_idw,alta,NaN,NaN,3.0,94.946721,25.089393,estimado_idw,alta,NaN,NaN,3.0,94.946721,60.271288,estimado_idw,alta,NaN,NaN,3.0,94.946721,0,estimado_tres_variaveis,4,alta
1,4316204,Rondinha,RS,sul,2019,1606.290803,estimado_idw,alta,NaN,NaN,3.0,65.019557,18.880873,estimado_idw,alta,NaN,NaN,3.0,65.019557,76.405028,estimado_idw,alta,NaN,NaN,3.0,65.019557,0,estimado_tres_variaveis,4,alta
2,4317558,Santo Antônio do Palma,RS,sul,2019,1667.721094,estimado_idw,alta,NaN,NaN,3.0,57.274133,18.313975,estimado_idw,alta,NaN,NaN,3.0,57.274133,78.396824,estimado_idw,alta,NaN,NaN,3.0,57.274133,0,estimado_tres_variaveis,4,alta
3,4209508,Laurentino,SC,sul,2019,1302.575419,estimado_idw,alta,NaN,NaN,3.0,86.066456,19.067521,estimado_idw,alta,NaN,NaN,3.0,86.066456,83.134326,estimado_idw,alta,NaN,NaN,3.0,86.066456,0,estimado_tres_variaveis,4,alta
4,4202107,Barra Velha,SC,sul,2019,1528.704529,estimado_idw,alta,NaN,NaN,3.0,96.733362,20.708063,estimado_idw,alta,NaN,NaN,3.0,96.733362,83.794815,estimado_idw,alta,NaN,NaN,3.0,96.733362,0,estimado_tres_variaveis,4,alta



MAPBIOMAS COBERTURA


,codigo_ibge,municipio,uf,regiao,ano,quantidade_biomas,biomas_presentes,area_total_mapeada_ha,area_ibge_ha,diferenca_area_pct,diferenca_area_abs_pct,faixa_diferenca_area_ibge,area_floresta_ha,area_formacao_natural_nao_florestal_ha,area_cobertura_natural_ha,area_agropecuaria_ha,area_pastagem_ha,area_agricultura_ha,area_lavoura_temporaria_ha,area_soja_mapbiomas_ha,area_silvicultura_ha,area_mosaico_usos_ha,area_nao_vegetada_ha,area_agua_natural_ha,area_aquicultura_ha,area_nao_observada_ha,pct_floresta,pct_formacao_natural_nao_florestal,pct_cobertura_natural,pct_agropecuaria,pct_pastagem,pct_agricultura,pct_lavoura_temporaria,pct_soja_mapbiomas,pct_silvicultura,pct_mosaico_usos,pct_area_nao_vegetada,pct_agua_natural,pct_aquicultura,pct_nao_observada,pct_fechamento_territorial
0,4100103,Abatiá,PR,Sul,2019,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2110.560484,20.153876,2130.714359,20501.078533,5773.710590,8494.500847,8183.880715,6337.048921,155.260906,6077.606191,159.532255,80.725196,0.0,0.0,9.227684,0.088116,9.315800,89.633759,25.243520,37.139219,35.781142,27.706519,0.678824,26.572197,0.697499,0.352943,0.0,0.0,100.0
1,4100103,Abatiá,PR,Sul,2020,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2138.038478,24.102157,2162.140635,20465.703400,5515.560753,8557.179526,8245.983106,6814.108692,153.615340,6239.347781,163.892775,80.313534,0.0,0.0,9.347822,0.105378,9.453200,89.479094,24.114851,37.413259,36.052662,29.792295,0.671629,27.279355,0.716564,0.351143,0.0,0.0,100.0
2,4100103,Abatiá,PR,Sul,2021,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2144.043562,24.842192,2168.885754,20457.066441,5136.433064,8573.384341,8262.928519,6855.572934,153.039484,6594.209551,165.620277,80.477873,0.0,0.0,9.374077,0.108614,9.482691,89.441332,22.457248,37.484109,36.126750,29.973583,0.669111,28.830863,0.724116,0.351861,0.0,0.0,100.0
3,4100103,Abatiá,PR,Sul,2022,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2120.430937,27.228733,2147.659670,20463.566371,4904.415611,8601.355462,8291.722262,6969.448836,152.628175,6805.167123,176.562297,84.262006,0.0,0.0,9.270839,0.119048,9.389887,89.469750,21.442833,37.606403,36.252641,30.471465,0.667313,29.753201,0.771957,0.368406,0.0,0.0,100.0
4,4100103,Abatiá,PR,Sul,2023,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2111.213707,26.652856,2137.866563,20467.189193,4644.000071,8617.563791,8307.766021,7127.089229,152.134601,7053.490730,180.100145,86.894442,0.0,0.0,9.230540,0.116530,9.347070,89.485590,20.304258,37.677268,36.322787,31.160692,0.665155,30.838909,0.787425,0.379915,0.0,0.0,100.0


In [8]:
# ============================================================
# PADRONIZAÇÃO DAS CHAVES DE INTEGRAÇÃO
# ============================================================

BASES_INTEGRACAO = {
    "pam_solo": pam_solo,
    "inmet": inmet,
    "mapbiomas_cobertura": mapbiomas_cobertura
}


for nome_base, base in BASES_INTEGRACAO.items():

    base[
        "codigo_ibge"
    ] = (
        base[
            "codigo_ibge"
        ]
        .astype("string")
        .str.strip()
        .str.zfill(7)
    )

    base[
        "ano"
    ] = pd.to_numeric(
        base[
            "ano"
        ],
        errors="raise"
    ).astype(int)


print(
    "Chaves codigo_ibge + ano padronizadas."
)

Chaves codigo_ibge + ano padronizadas.


In [9]:
# ============================================================
# VALIDAÇÃO DE ESCALA — VARIÁVEIS CLIMÁTICAS
# ============================================================

COLUNAS_ESCALA_INMET = [
    "precipitacao_anual_mm",
    "distancia_max_idw_precipitacao_km",
    "temperatura_media_anual_c",
    "distancia_max_idw_temperatura_km",
    "umidade_media_anual_pct",
    "distancia_max_idw_umidade_km"
]


display(
    inmet[
        COLUNAS_ESCALA_INMET
    ]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
precipitacao_anual_mm,9966.0,1542.578403,427.858376,348.400000,1233.537070,1464.514595,1805.623592,3394.200000
distancia_max_idw_precipitacao_km,9282.0,112.076880,61.099063,24.285074,70.850444,93.277640,136.788083,697.106092
temperatura_media_anual_c,9966.0,20.986132,2.870842,10.634183,18.763334,20.282317,23.595863,29.354010
distancia_max_idw_temperatura_km,9191.0,105.869902,56.621737,24.829939,68.770640,89.334055,126.648664,634.687142
umidade_media_anual_pct,9966.0,72.348863,7.041077,53.208944,66.538527,73.163060,77.491446,92.745039
distancia_max_idw_umidade_km,9250.0,109.332650,56.320221,30.390782,72.227715,92.777412,129.692495,634.687142


In [10]:
display(
    inmet[
        [
            "codigo_ibge",
            "municipio",
            "ano"
        ]
        +
        COLUNAS_ESCALA_INMET
    ]
    .head(10)
)

,codigo_ibge,municipio,ano,precipitacao_anual_mm,distancia_max_idw_precipitacao_km,temperatura_media_anual_c,distancia_max_idw_temperatura_km,umidade_media_anual_pct,distancia_max_idw_umidade_km
0,5219902,São Francisco de Goiás,2019,1073.612878,94.946721,25.089393,94.946721,60.271288,94.946721
1,4316204,Rondinha,2019,1606.290803,65.019557,18.880873,65.019557,76.405028,65.019557
2,4317558,Santo Antônio do Palma,2019,1667.721094,57.274133,18.313975,57.274133,78.396824,57.274133
3,4209508,Laurentino,2019,1302.575419,86.066456,19.067521,86.066456,83.134326,86.066456
4,4202107,Barra Velha,2019,1528.704529,96.733362,20.708063,96.733362,83.794815,96.733362
5,4217808,Taió,2019,1412.927905,80.642221,17.983916,80.642221,81.305296,80.642221
6,4319307,São Paulo das Missões,2019,1870.235128,125.649534,20.801086,117.563585,72.301902,117.563585
7,5215652,Palestina de Goiás,2019,1248.582639,112.028074,24.570169,112.028074,63.721447,112.028074
8,4311429,Lajeado do Bugre,2019,1595.735634,60.054824,19.579546,60.054824,74.732268,60.054824
9,4318002,São Borja,2019,1632.800000,NaN,20.954209,NaN,73.057434,NaN


In [11]:
# ============================================================
# AUDITORIA DAS CHAVES DAS BASES
# ============================================================

CHAVE = [
    "codigo_ibge",
    "ano"
]


def auditar_base(
    nome,
    base
):

    print(
        "\n" + "=" * 70
    )

    print(
        nome
    )

    print(
        "=" * 70
    )

    print(
        "Linhas:",
        len(base)
    )

    print(
        "Municípios/unidades territoriais:",
        base[
            "codigo_ibge"
        ].nunique()
    )

    print(
        "Anos:",
        sorted(
            base[
                "ano"
            ]
            .unique()
            .tolist()
        )
    )

    print(
        "Chaves nulas:",
        base[
            CHAVE
        ]
        .isna()
        .any(axis=1)
        .sum()
    )

    print(
        "Duplicatas codigo_ibge + ano:",
        base
        .duplicated(
            subset=CHAVE
        )
        .sum()
    )

    print(
        "\nRegistros por ano:"
    )

    display(
        base[
            "ano"
        ]
        .value_counts()
        .sort_index()
    )


auditar_base(
    "PAM + SOLO",
    pam_solo
)

auditar_base(
    "INMET",
    inmet
)

auditar_base(
    "MAPBIOMAS COBERTURA",
    mapbiomas_cobertura
)


PAM + SOLO
Linhas: 8674
Municípios/unidades territoriais: 1505
Anos: [2019, 2020, 2021, 2022, 2023, 2024]
Chaves nulas: 0
Duplicatas codigo_ibge + ano: 0

Registros por ano:


ano
2019    1411
2020    1421
2021    1430
2022    1456
2023    1474
2024    1482
Name: count, dtype: int64


INMET
Linhas: 9966
Municípios/unidades territoriais: 1661
Anos: [2019, 2020, 2021, 2022, 2023, 2024]
Chaves nulas: 0
Duplicatas codigo_ibge + ano: 0

Registros por ano:


ano
2019    1661
2020    1661
2021    1661
2022    1661
2023    1661
2024    1661
Name: count, dtype: int64


MAPBIOMAS COBERTURA
Linhas: 9960
Municípios/unidades territoriais: 1660
Anos: [2019, 2020, 2021, 2022, 2023, 2024]
Chaves nulas: 0
Duplicatas codigo_ibge + ano: 0

Registros por ano:


ano
2019    1660
2020    1660
2021    1660
2022    1660
2023    1660
2024    1660
Name: count, dtype: int64

In [12]:
# ============================================================
# UNIVERSO ANALÍTICO — PAM + SOLO
# ============================================================

print(
    "Culturas:"
)

display(
    pam_solo[
        "cultura"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nStatus dos dados de solo:"
)

display(
    pam_solo[
        "status_dado"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nMunicípios distintos:"
)

print(
    pam_solo[
        "codigo_ibge"
    ].nunique()
)


anos_por_municipio_pam = (
    pam_solo
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf"
        ],
        as_index=False
    )
    .agg(
        quantidade_anos=(
            "ano",
            "nunique"
        )
    )
)


print(
    "\nQuantidade de anos disponíveis por município:"
)

display(
    anos_por_municipio_pam[
        "quantidade_anos"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nMunicípios presentes nos 6 anos:"
)

print(
    (
        anos_por_municipio_pam[
            "quantidade_anos"
        ] == 6
    ).sum()
)

Culturas:


cultura
soja    8674
Name: count, dtype: int64


Status dos dados de solo:


status_dado
disponivel    8674
Name: count, dtype: int64


Municípios distintos:
1505

Quantidade de anos disponíveis por município:


quantidade_anos
1      24
2      27
3      25
4      15
5      23
6    1391
Name: count, dtype: int64


Municípios presentes nos 6 anos:
1391


In [13]:
# ============================================================
# COBERTURA DAS CHAVES ENTRE AS BASES
# ============================================================

chaves_pam = set(
    map(
        tuple,
        pam_solo[
            CHAVE
        ].to_numpy()
    )
)


chaves_inmet = set(
    map(
        tuple,
        inmet[
            CHAVE
        ].to_numpy()
    )
)


chaves_cobertura = set(
    map(
        tuple,
        mapbiomas_cobertura[
            CHAVE
        ].to_numpy()
    )
)


pam_sem_inmet = (
    chaves_pam
    -
    chaves_inmet
)


pam_sem_cobertura = (
    chaves_pam
    -
    chaves_cobertura
)


print(
    "Chaves PAM + Solo:"
)

print(
    len(
        chaves_pam
    )
)


print(
    "\nPAM sem correspondência INMET:"
)

print(
    len(
        pam_sem_inmet
    )
)


print(
    "\nPAM sem correspondência MapBiomas Cobertura:"
)

print(
    len(
        pam_sem_cobertura
    )
)

Chaves PAM + Solo:
8674

PAM sem correspondência INMET:
0

PAM sem correspondência MapBiomas Cobertura:
0


In [14]:
# ============================================================
# DETALHE DE EVENTUAIS AUSÊNCIAS
# ============================================================

if len(
    pam_sem_inmet
) > 0:

    print(
        "\nPAM sem INMET:"
    )

    display(
        pam_solo[
            pam_solo[
                CHAVE
            ]
            .apply(
                tuple,
                axis=1
            )
            .isin(
                pam_sem_inmet
            )
        ][
            [
                "codigo_ibge",
                "municipio",
                "uf",
                "ano"
            ]
        ]
        .drop_duplicates()
    )


if len(
    pam_sem_cobertura
) > 0:

    print(
        "\nPAM sem MapBiomas Cobertura:"
    )

    display(
        pam_solo[
            pam_solo[
                CHAVE
            ]
            .apply(
                tuple,
                axis=1
            )
            .isin(
                pam_sem_cobertura
            )
        ][
            [
                "codigo_ibge",
                "municipio",
                "uf",
                "ano"
            ]
        ]
        .drop_duplicates()
    )

In [15]:
# ============================================================
# SELEÇÃO ANALÍTICA — INMET
# ============================================================

COLUNAS_INMET_ANALITICAS = [
    # Chave
    "codigo_ibge",
    "ano",

    # Indicadores climáticos
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct",

    # Origem
    "origem_precipitacao",
    "origem_temperatura",
    "origem_umidade",

    # Qualidade espacial
    "qualidade_espacial_precipitacao",
    "qualidade_espacial_temperatura",
    "qualidade_espacial_umidade",

    # Qualidade consolidada
    "tipo_representacao_climatica",
    "score_qualidade_climatica",
    "qualidade_climatica_geral"
]


inmet_analitico = (
    inmet[
        COLUNAS_INMET_ANALITICAS
    ]
    .copy()
)


print(
    "Dimensão INMET completo:"
)

print(
    inmet.shape
)


print(
    "\nDimensão INMET analítico:"
)

print(
    inmet_analitico.shape
)


print(
    "\nDuplicatas:"
)

print(
    inmet_analitico
    .duplicated(
        subset=CHAVE
    )
    .sum()
)

Dimensão INMET completo:
(9966, 30)

Dimensão INMET analítico:
(9966, 14)

Duplicatas:
0


In [16]:
# ============================================================
# SELEÇÃO ANALÍTICA — MAPBIOMAS COBERTURA
# ============================================================

COLUNAS_COBERTURA_ANALITICAS = [
    # Chave
    "codigo_ibge",
    "ano",

    # Contexto territorial
    "quantidade_biomas",
    "biomas_presentes",
    "area_total_mapeada_ha",

    # Cobertura natural
    "area_cobertura_natural_ha",
    "pct_cobertura_natural",
    "area_floresta_ha",
    "pct_floresta",

    # Uso agropecuário
    "area_agropecuaria_ha",
    "pct_agropecuaria",
    "area_pastagem_ha",
    "pct_pastagem",
    "area_agricultura_ha",
    "pct_agricultura",

    # Soja
    "area_soja_mapbiomas_ha",
    "pct_soja_mapbiomas",

    # Controle territorial
    "diferenca_area_abs_pct",
    "faixa_diferenca_area_ibge"
]


cobertura_analitica = (
    mapbiomas_cobertura[
        COLUNAS_COBERTURA_ANALITICAS
    ]
    .copy()
)


print(
    "Dimensão MapBiomas completo:"
)

print(
    mapbiomas_cobertura.shape
)


print(
    "\nDimensão MapBiomas analítico:"
)

print(
    cobertura_analitica.shape
)


print(
    "\nDuplicatas:"
)

print(
    cobertura_analitica
    .duplicated(
        subset=CHAVE
    )
    .sum()
)

Dimensão MapBiomas completo:
(9960, 41)

Dimensão MapBiomas analítico:
(9960, 19)

Duplicatas:
0


In [17]:
# ============================================================
# VALIDAÇÃO DOS SUBCONJUNTOS ANALÍTICOS
# ============================================================

print(
    "PAM + Solo:"
)

print(
    pam_solo.shape
)


print(
    "\nINMET analítico:"
)

print(
    inmet_analitico.shape
)


print(
    "\nCobertura analítica:"
)

print(
    cobertura_analitica.shape
)


print(
    "\nDuplicatas:"
)

print(
    "PAM + Solo:",
    pam_solo.duplicated(
        subset=CHAVE
    ).sum()
)

print(
    "INMET:",
    inmet_analitico.duplicated(
        subset=CHAVE
    ).sum()
)

print(
    "Cobertura:",
    cobertura_analitica.duplicated(
        subset=CHAVE
    ).sum()
)

PAM + Solo:
(8674, 16)

INMET analítico:
(9966, 14)

Cobertura analítica:
(9960, 19)

Duplicatas:
PAM + Solo: 0
INMET: 0
Cobertura: 0


In [18]:
# ============================================================
# INTEGRAÇÃO 1 — PAM + SOLO × INMET
# ============================================================

base_agroambiental = (
    pam_solo
    .merge(
        inmet_analitico,
        on=CHAVE,
        how="left",
        validate="one_to_one"
    )
)


print(
    "Dimensão após integração com INMET:"
)

print(
    base_agroambiental.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_agroambiental
    .duplicated(
        subset=CHAVE
    )
    .sum()
)


print(
    "\nNulos nas variáveis climáticas principais:"
)

display(
    base_agroambiental[
        [
            "precipitacao_anual_mm",
            "temperatura_media_anual_c",
            "umidade_media_anual_pct"
        ]
    ]
    .isna()
    .sum()
)

Dimensão após integração com INMET:
(8674, 28)

Duplicatas codigo_ibge + ano:
0

Nulos nas variáveis climáticas principais:


precipitacao_anual_mm        0
temperatura_media_anual_c    0
umidade_media_anual_pct      0
dtype: int64

In [19]:
# ============================================================
# INTEGRAÇÃO 2 — BASE AGROAMBIENTAL × MAPBIOMAS COBERTURA
# ============================================================

base_agroambiental = (
    base_agroambiental
    .merge(
        cobertura_analitica,
        on=CHAVE,
        how="left",
        validate="one_to_one"
    )
)


print(
    "Dimensão após integração com MapBiomas Cobertura:"
)

print(
    base_agroambiental.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_agroambiental
    .duplicated(
        subset=CHAVE
    )
    .sum()
)

Dimensão após integração com MapBiomas Cobertura:
(8674, 45)

Duplicatas codigo_ibge + ano:
0


In [20]:
# ============================================================
# VALIDAÇÃO DE COMPLETUDE APÓS OS MERGES
# ============================================================

COLUNAS_PRINCIPAIS_INTEGRACAO = [
    "carbono_solo_t_ha",
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct",
    "area_cobertura_natural_ha",
    "pct_cobertura_natural",
    "area_agropecuaria_ha",
    "pct_agropecuaria",
    "area_soja_mapbiomas_ha",
    "pct_soja_mapbiomas"
]


print(
    "Nulos nas principais variáveis integradas:"
)

display(
    base_agroambiental[
        COLUNAS_PRINCIPAIS_INTEGRACAO
    ]
    .isna()
    .sum()
)

Nulos nas principais variáveis integradas:


carbono_solo_t_ha            0
precipitacao_anual_mm        0
temperatura_media_anual_c    0
umidade_media_anual_pct      0
area_cobertura_natural_ha    0
pct_cobertura_natural        0
area_agropecuaria_ha         0
pct_agropecuaria             0
area_soja_mapbiomas_ha       0
pct_soja_mapbiomas           0
dtype: int64

In [21]:
# ============================================================
# INSPEÇÃO DA BASE INTEGRADA
# ============================================================

print(
    "Dimensão:"
)

print(
    base_agroambiental.shape
)


print(
    "\nMunicípios:"
)

print(
    base_agroambiental[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nAnos:"
)

print(
    sorted(
        base_agroambiental[
            "ano"
        ]
        .unique()
        .tolist()
    )
)


print(
    "\nColunas:"
)

print(
    base_agroambiental.columns.tolist()
)


display(
    base_agroambiental.head()
)

Dimensão:
(8674, 45)

Municípios:
1505

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]

Colunas:
['codigo_ibge', 'municipio', 'uf', 'regiao', 'ano', 'cultura', 'area_plantada_ha', 'area_colhida_ha', 'area_nao_colhida_ha', 'aproveitamento_area_pct', 'quantidade_produzida_t', 'rendimento_medio_kg_ha', 'valor_producao_mil_reais', 'area_km2', 'carbono_solo_t_ha', 'status_dado', 'precipitacao_anual_mm', 'temperatura_media_anual_c', 'umidade_media_anual_pct', 'origem_precipitacao', 'origem_temperatura', 'origem_umidade', 'qualidade_espacial_precipitacao', 'qualidade_espacial_temperatura', 'qualidade_espacial_umidade', 'tipo_representacao_climatica', 'score_qualidade_climatica', 'qualidade_climatica_geral', 'quantidade_biomas', 'biomas_presentes', 'area_total_mapeada_ha', 'area_cobertura_natural_ha', 'pct_cobertura_natural', 'area_floresta_ha', 'pct_floresta', 'area_agropecuaria_ha', 'pct_agropecuaria', 'area_pastagem_ha', 'pct_pastagem', 'area_agricultura_ha', 'pct_agricultura', 'area_soja_mapbi

,codigo_ibge,municipio,uf,regiao,ano,cultura,area_plantada_ha,area_colhida_ha,area_nao_colhida_ha,aproveitamento_area_pct,quantidade_produzida_t,rendimento_medio_kg_ha,valor_producao_mil_reais,area_km2,carbono_solo_t_ha,status_dado,precipitacao_anual_mm,temperatura_media_anual_c,umidade_media_anual_pct,origem_precipitacao,origem_temperatura,origem_umidade,qualidade_espacial_precipitacao,qualidade_espacial_temperatura,qualidade_espacial_umidade,tipo_representacao_climatica,score_qualidade_climatica,qualidade_climatica_geral,quantidade_biomas,biomas_presentes,area_total_mapeada_ha,area_cobertura_natural_ha,pct_cobertura_natural,area_floresta_ha,pct_floresta,area_agropecuaria_ha,pct_agropecuaria,area_pastagem_ha,pct_pastagem,area_agricultura_ha,pct_agricultura,area_soja_mapbiomas_ha,pct_soja_mapbiomas,diferenca_area_abs_pct,faixa_diferenca_area_ibge
0,4100103,Abatiá,PR,Sul,2019,soja,10570.0,10570.0,0.0,100.0,36784.0,3480.0,43387.0,228.717,55.173093,disponivel,994.780491,21.908078,69.258447,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2130.714359,9.315800,2110.560484,9.227684,20501.078533,89.633759,5773.710590,25.243520,8494.500847,37.139219,6337.048921,27.706519,0.001532,ate_0_1_pct
1,4100103,Abatiá,PR,Sul,2020,soja,10470.0,10470.0,0.0,100.0,36435.0,3480.0,50268.0,228.717,55.169301,disponivel,907.189175,21.716718,67.530900,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2162.140635,9.453200,2138.038478,9.347822,20465.703400,89.479094,5515.560753,24.114851,8557.179526,37.413259,6814.108692,29.792295,0.001532,ate_0_1_pct
2,4100103,Abatiá,PR,Sul,2021,soja,10220.0,10220.0,0.0,100.0,27727.0,2713.0,69682.0,228.717,55.039162,disponivel,1219.103697,21.308696,64.969887,estimado_idw,estimado_idw,estimado_idw,baixa,media,media,estimado_tres_variaveis,2,baixa,1,Mata Atlântica,22872.050344,2168.885754,9.482691,2144.043562,9.374077,20457.066441,89.441332,5136.433064,22.457248,8573.384341,37.484109,6855.572934,29.973583,0.001532,ate_0_1_pct
3,4100103,Abatiá,PR,Sul,2022,soja,10470.0,10470.0,0.0,100.0,37064.0,3540.0,114457.0,228.717,55.024834,disponivel,1501.673064,20.010075,71.960374,estimado_idw,estimado_idw,estimado_idw,baixa,baixa,baixa,estimado_tres_variaveis,2,baixa,1,Mata Atlântica,22872.050344,2147.659670,9.389887,2120.430937,9.270839,20463.566371,89.469750,4904.415611,21.442833,8601.355462,37.606403,6969.448836,30.471465,0.001532,ate_0_1_pct
4,4100103,Abatiá,PR,Sul,2023,soja,9960.0,9960.0,0.0,100.0,35856.0,3600.0,94498.0,228.717,55.025820,disponivel,1697.612945,20.633306,74.495469,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2137.866563,9.347070,2111.213707,9.230540,20467.189193,89.485590,4644.000071,20.304258,8617.563791,37.677268,7127.089229,31.160692,0.001532,ate_0_1_pct


In [23]:
# ============================================================
# VALIDAÇÃO FINAL — BASE AGROAMBIENTAL INTEGRADA
# ============================================================

COLUNAS_CHAVE_IDENTIFICACAO = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "ano",
    "cultura"
]

COLUNAS_PRINCIPAIS = [
    # Agricultura
    "area_plantada_ha",
    "area_colhida_ha",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha",

    # Solo
    "carbono_solo_t_ha",

    # Clima
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct",

    # Cobertura
    "area_cobertura_natural_ha",
    "pct_cobertura_natural",
    "area_agropecuaria_ha",
    "pct_agropecuaria",
    "area_soja_mapbiomas_ha",
    "pct_soja_mapbiomas"
]


print("Dimensão:")
print(base_agroambiental.shape)

print("\nDuplicatas codigo_ibge + ano:")
print(
    base_agroambiental
    .duplicated(
        subset=CHAVE
    )
    .sum()
)

print("\nNulos nas chaves/identificação:")
display(
    base_agroambiental[
        COLUNAS_CHAVE_IDENTIFICACAO
    ]
    .isna()
    .sum()
)

print("\nNulos nas variáveis principais:")
display(
    base_agroambiental[
        COLUNAS_PRINCIPAIS
    ]
    .isna()
    .sum()
)

Dimensão:
(8674, 45)

Duplicatas codigo_ibge + ano:
0

Nulos nas chaves/identificação:


codigo_ibge    0
municipio      0
uf             0
regiao         0
ano            0
cultura        0
dtype: int64


Nulos nas variáveis principais:


area_plantada_ha             0
area_colhida_ha              5
quantidade_produzida_t       5
rendimento_medio_kg_ha       5
carbono_solo_t_ha            0
precipitacao_anual_mm        0
temperatura_media_anual_c    0
umidade_media_anual_pct      0
area_cobertura_natural_ha    0
pct_cobertura_natural        0
area_agropecuaria_ha         0
pct_agropecuaria             0
area_soja_mapbiomas_ha       0
pct_soja_mapbiomas           0
dtype: int64

In [24]:
# ============================================================
# VALIDAÇÃO DE DOMÍNIO
# ============================================================

print("Cultura:")
display(
    base_agroambiental[
        "cultura"
    ]
    .value_counts(
        dropna=False
    )
)

print("\nUFs:")
display(
    base_agroambiental[
        "uf"
    ]
    .value_counts()
    .sort_index()
)

print("\nRegiões:")
display(
    base_agroambiental[
        "regiao"
    ]
    .value_counts()
)

print("\nAnos:")
display(
    base_agroambiental[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

Cultura:


cultura
soja    8674
Name: count, dtype: int64


UFs:


uf
DF       6
GO    1303
MS     463
MT     761
PR    2311
RS    2574
SC    1256
Name: count, dtype: int64


Regiões:


regiao
Sul             6141
Centro-Oeste    2533
Name: count, dtype: int64


Anos:


ano
2019    1411
2020    1421
2021    1430
2022    1456
2023    1474
2024    1482
Name: count, dtype: int64

In [25]:
# ============================================================
# ORGANIZAÇÃO FINAL
# ============================================================

base_agroambiental = (
    base_agroambiental
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    base_agroambiental.shape
)

display(
    base_agroambiental.head()
)

(8674, 45)


,codigo_ibge,municipio,uf,regiao,ano,cultura,area_plantada_ha,area_colhida_ha,area_nao_colhida_ha,aproveitamento_area_pct,quantidade_produzida_t,rendimento_medio_kg_ha,valor_producao_mil_reais,area_km2,carbono_solo_t_ha,status_dado,precipitacao_anual_mm,temperatura_media_anual_c,umidade_media_anual_pct,origem_precipitacao,origem_temperatura,origem_umidade,qualidade_espacial_precipitacao,qualidade_espacial_temperatura,qualidade_espacial_umidade,tipo_representacao_climatica,score_qualidade_climatica,qualidade_climatica_geral,quantidade_biomas,biomas_presentes,area_total_mapeada_ha,area_cobertura_natural_ha,pct_cobertura_natural,area_floresta_ha,pct_floresta,area_agropecuaria_ha,pct_agropecuaria,area_pastagem_ha,pct_pastagem,area_agricultura_ha,pct_agricultura,area_soja_mapbiomas_ha,pct_soja_mapbiomas,diferenca_area_abs_pct,faixa_diferenca_area_ibge
0,4100103,Abatiá,PR,Sul,2019,soja,10570.0,10570.0,0.0,100.0,36784.0,3480.0,43387.0,228.717,55.173093,disponivel,994.780491,21.908078,69.258447,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2130.714359,9.315800,2110.560484,9.227684,20501.078533,89.633759,5773.710590,25.243520,8494.500847,37.139219,6337.048921,27.706519,0.001532,ate_0_1_pct
1,4100103,Abatiá,PR,Sul,2020,soja,10470.0,10470.0,0.0,100.0,36435.0,3480.0,50268.0,228.717,55.169301,disponivel,907.189175,21.716718,67.530900,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2162.140635,9.453200,2138.038478,9.347822,20465.703400,89.479094,5515.560753,24.114851,8557.179526,37.413259,6814.108692,29.792295,0.001532,ate_0_1_pct
2,4100103,Abatiá,PR,Sul,2021,soja,10220.0,10220.0,0.0,100.0,27727.0,2713.0,69682.0,228.717,55.039162,disponivel,1219.103697,21.308696,64.969887,estimado_idw,estimado_idw,estimado_idw,baixa,media,media,estimado_tres_variaveis,2,baixa,1,Mata Atlântica,22872.050344,2168.885754,9.482691,2144.043562,9.374077,20457.066441,89.441332,5136.433064,22.457248,8573.384341,37.484109,6855.572934,29.973583,0.001532,ate_0_1_pct
3,4100103,Abatiá,PR,Sul,2022,soja,10470.0,10470.0,0.0,100.0,37064.0,3540.0,114457.0,228.717,55.024834,disponivel,1501.673064,20.010075,71.960374,estimado_idw,estimado_idw,estimado_idw,baixa,baixa,baixa,estimado_tres_variaveis,2,baixa,1,Mata Atlântica,22872.050344,2147.659670,9.389887,2120.430937,9.270839,20463.566371,89.469750,4904.415611,21.442833,8601.355462,37.606403,6969.448836,30.471465,0.001532,ate_0_1_pct
4,4100103,Abatiá,PR,Sul,2023,soja,9960.0,9960.0,0.0,100.0,35856.0,3600.0,94498.0,228.717,55.025820,disponivel,1697.612945,20.633306,74.495469,estimado_idw,estimado_idw,estimado_idw,alta,alta,alta,estimado_tres_variaveis,4,alta,1,Mata Atlântica,22872.050344,2137.866563,9.347070,2111.213707,9.230540,20467.189193,89.485590,4644.000071,20.304258,8617.563791,37.677268,7127.089229,31.160692,0.001532,ate_0_1_pct


In [26]:
# ============================================================
# EXPORTAÇÃO — BASE AGROAMBIENTAL INTEGRADA
# PAM + SOLO + INMET + MAPBIOMAS COBERTURA
# ============================================================

INTEGRACAO_CURATED_DIR = (
    CURATED_DIR
    / "integracao_agroambiental"
)


INTEGRACAO_CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_BASE_AGROAMBIENTAL = (
    INTEGRACAO_CURATED_DIR
    / "base_agroambiental_pam_solo_inmet_cobertura_soja_centro_oeste_sul_2019_2024.csv"
)


base_agroambiental.to_csv(
    ARQUIVO_BASE_AGROAMBIENTAL,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Arquivo salvo:"
)

print(
    ARQUIVO_BASE_AGROAMBIENTAL
)

print(
    "\nDimensão:"
)

print(
    base_agroambiental.shape
)

Arquivo salvo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\integracao_agroambiental\base_agroambiental_pam_solo_inmet_cobertura_soja_centro_oeste_sul_2019_2024.csv

Dimensão:
(8674, 45)


In [27]:
# ============================================================
# VALIDAÇÃO DE RELEITURA
# ============================================================

base_agroambiental_check = pd.read_csv(
    ARQUIVO_BASE_AGROAMBIENTAL,
    dtype={
        "codigo_ibge": "string"
    }
)


base_agroambiental_check[
    "codigo_ibge"
] = (
    base_agroambiental_check[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print("Dimensão:")
print(base_agroambiental_check.shape)

print("\nDuplicatas:")
print(
    base_agroambiental_check
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)

print("\nMunicípios:")
print(
    base_agroambiental_check[
        "codigo_ibge"
    ]
    .nunique()
)

print("\nAnos:")
print(
    sorted(
        base_agroambiental_check[
            "ano"
        ]
        .unique()
        .tolist()
    )
)

Dimensão:
(8674, 45)

Duplicatas:
0

Municípios:
1505

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]


In [28]:
# ============================================================
# AUDITORIA FINAL — REGISTROS PAM COM VALORES AUSENTES
# ============================================================

COLUNAS_PAM_COM_AUSENCIA = [
    "area_colhida_ha",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha"
]


registros_pam_com_ausencia = (
    base_agroambiental[
        base_agroambiental[
            COLUNAS_PAM_COM_AUSENCIA
        ]
        .isna()
        .any(axis=1)
    ]
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",
            "area_plantada_ha",
            "area_colhida_ha",
            "quantidade_produzida_t",
            "rendimento_medio_kg_ha"
        ]
    ]
)


print(
    "Quantidade de registros PAM com ausência:"
)

print(
    len(registros_pam_com_ausencia)
)


display(
    registros_pam_com_ausencia
)

Quantidade de registros PAM com ausência:
5


,codigo_ibge,municipio,uf,ano,area_plantada_ha,area_colhida_ha,quantidade_produzida_t,rendimento_medio_kg_ha
1612,4120200,Porto Rico,PR,2024,150.0,NaN,NaN,NaN
3977,4304606,Canoas,RS,2024,150.0,NaN,NaN,NaN
4127,4305454,Cidreira,RS,2024,500.0,NaN,NaN,NaN
4627,4310330,Imbé,RS,2024,150.0,NaN,NaN,NaN
5117,4314050,Parobé,RS,2020,17.0,NaN,NaN,NaN


# Conclusão — Integração das Bases Curated

O Notebook 07 consolidou as principais bases agroambientais já tratadas
para o escopo de soja nas regiões Centro-Oeste e Sul, no período de
2019 a 2024.

## Fontes integradas

A base consolidada reúne:

- IBGE/PAM — produção municipal de soja;
- MapBiomas Solo — carbono orgânico do solo;
- INMET — precipitação, temperatura, umidade e indicadores de qualidade;
- MapBiomas Cobertura — cobertura natural, uso agropecuário, pastagem,
  agricultura e área classificada como soja.

## Chave de integração

A integração foi realizada por:

- `codigo_ibge`
- `ano`

O universo analítico foi definido a partir da base PAM + MapBiomas Solo,
representando observações município-ano com registro de produção de soja.

## Resultado

A base integrada possui:

- 8.674 observações município-ano;
- 1.505 municípios;
- período de 2019 a 2024;
- 45 variáveis;
- 0 duplicatas em `codigo_ibge + ano`;
- 100% de correspondência das observações PAM com INMET;
- 100% de correspondência das observações PAM com MapBiomas Cobertura.

As variáveis principais de solo, clima e cobertura utilizadas na integração
não apresentam valores ausentes.

Foram preservados valores ausentes existentes em registros específicos da
PAM, sem substituição artificial por zero.

## Observação metodológica

A base construída é uma integração agroambiental intermediária e não
representa, por si só, estimativa ou certificação de créditos de carbono.

As bases SEEG e EMBRAPA/BRLUC serão avaliadas separadamente antes de uma
integração posterior.

O arquivo consolidado foi exportado em:

`data/databases_curated/integracao_agroambiental/base_agroambiental_pam_solo_inmet_cobertura_soja_centro_oeste_sul_2019_2024.csv`